In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import joblib
import requests


# **Data Wrangling**

## **Gathering Data**

In [ ]:
file_id = '1d7j3-vot2e1NOVCMUvcKjmS35V_4plwx'
url     = f'https://drive.google.com/uc?id={file_id}&export=download'

response = requests.get(url)
with open('bahan_clean.xlsx', 'wb') as f:
    f.write(response.content)
xls = pd.read_excel('bahan_clean.xlsx', sheet_name=None)

df_product     = xls['Product']
df_translate   = xls['Product_Translate']
df_category    = xls['Category']

## **Assessing Data**

Mengevaluasi kualitas dari data
- **Tipe data** — apakah tipe data sudah sesuai
- **Missing value** — apakah ada nilai yang hilang
- **Invalid value** — apakah ada nilai error seperti `#VALUE!` dari Excel
- **Duplicate data** — apakah ada data yang terduplikasi
- **Inconsistent value** — apakah ada nilai yang tidak konsisten (misal satuan metrik yang berbeda)

### Assessing - df_product

In [ ]:
df_product.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 196 entries, 0 to 195
Data columns (total 38 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   ID                                196 non-null    int64  
 1   Category_ID                       196 non-null    int64  
 2   Name                              196 non-null    object 
 3   Name_subtitle                     138 non-null    object 
 4   Keywords                          192 non-null    object 
 5   Pantry_Min                        108 non-null    object 
 6   Pantry_Max                        137 non-null    object 
 7   Pantry_Metric                     140 non-null    object 
 8   Pantry_tips                       111 non-null    object 
 9   DOP_Pantry_Min                    153 non-null    object 
 10  DOP_Pantry_Max                    152 non-null    object 
 11  DOP_Pantry_Metric                 153 non-null    object 
 12  DOP_Pant

Dataset `df_product` terdiri dari 196 baris dan 38 kolom. Kolom `ID`,
`Category_ID`, dan `Name` sudah lengkap (tidak ada missing value).
Namun kolom-kolom penyimpanan seperti `Pantry_Min`, `Pantry_Max`,
`Refrigerate_Min`, dan `Freeze_Min` banyak yang tidak terisi —
menandakan tidak semua bahan memiliki data lengkap di semua metode
penyimpanan. Selain itu, sebagian besar kolom bertipe `object`
padahal seharusnya numeric — disebabkan oleh adanya nilai tidak valid
seperti `#VALUE!` dan teks campuran di dalam kolom tersebut.
Tipe data akan dikonversi ke numeric di tahap Cleaning.

In [ ]:
#Missing Value
print(df_product.isna().sum())

ID                                    0
Category_ID                           0
Name                                  0
Name_subtitle                        58
Keywords                              4
Pantry_Min                           88
Pantry_Max                           59
Pantry_Metric                        56
Pantry_tips                          85
DOP_Pantry_Min                       43
DOP_Pantry_Max                       44
DOP_Pantry_Metric                    43
DOP_Pantry_tips                      98
Pantry_After_Opening_Min             90
Pantry_After_Opening_Max             91
Pantry_After_Opening_Metric          89
Refrigerate_Min                      81
Refrigerate_Max                      80
Refrigerate_Metric                   77
Refrigerate_tips                     90
DOP_Refrigerate_Min                  35
DOP_Refrigerate_Max                  35
DOP_Refrigerate_Metric               35
DOP_Refrigerate_tips                 98
Refrigerate_After_Opening_Min        79


Terdapat missing value yang sangat tinggi pada kolom-kolom penyimpanan.
Hal ini wajar karena tidak semua bahan memiliki data untuk setiap
metode penyimpanan.

In [ ]:
# Invalid value
invalid_count = (df_product == '#VALUE!').sum().sum()
print(f'Jumlah #VALUE! (error Excel): {invalid_count}')
print('\n#VALUE! per kolom:')
invalid_per_col = (df_product == '#VALUE!').sum()
print(invalid_per_col[invalid_per_col > 0])

Jumlah #VALUE! (error Excel): 2645

#VALUE! per kolom:
Name_subtitle                       98
Keywords                            97
Pantry_Min                           5
Pantry_Max                          78
Pantry_Metric                        5
Pantry_tips                         97
DOP_Pantry_Min                      94
DOP_Pantry_Max                      94
DOP_Pantry_Metric                   94
DOP_Pantry_tips                     98
Pantry_After_Opening_Min            97
Pantry_After_Opening_Max            98
Pantry_After_Opening_Metric         97
Refrigerate_Min                      3
Refrigerate_Max                     80
Refrigerate_Metric                   3
Refrigerate_tips                    97
DOP_Refrigerate_Min                 95
DOP_Refrigerate_Max                 95
DOP_Refrigerate_Metric              95
DOP_Refrigerate_tips                98
Refrigerate_After_Opening_Min       98
Refrigerate_After_Opening_Max       98
Refrigerate_After_Opening_Metric    98
Refrigera

Ditemukan **2.645 sel** berisi `#VALUE!` yang merupakan error formula
dari Excel dikarenakan menggunakan fungsi translate. Nilai ini tersebar di hampir semua kolom penyimpanan dan
Metric. Seluruhnya akan diganti dengan `NaN` di tahap Cleaning agar
tidak mengganggu proses konversi ke hari.

In [ ]:
print('=== Duplicate di df_product ===')
print(f'Duplicate rows (semua kolom): {df_product.duplicated().sum()}')
print(f'Duplicate ID                : {df_product["ID"].duplicated().sum()}')
print(f'Duplicate Name              : {df_product["Name"].duplicated().sum()}')
print()

dup_names = df_product[df_product['Name'].duplicated(keep=False)][['ID','Name']].sort_values('Name')
print(f'Detail Name duplikat ({len(dup_names)} baris):')
print(dup_names.to_string(index=False))

=== Duplicate di df_product ===
Duplicate rows (semua kolom): 5
Duplicate ID                : 27
Duplicate Name              : 24

Detail Name duplikat (39 baris):
 ID                      Name
 34                 Asparagus
 34                 Asparagus
 82               Basil leave
 82               Basil leave
 37                  Broccoli
 37                  Broccoli
 26               Brown sugar
 26               Brown sugar
 52                   Chayote
 96                   Chayote
 52                   Chayote
 92 Chicken Liver and Gizzard
 92 Chicken Liver and Gizzard
 74                  Cinnamon
 74                  Cinnamon
143             Coconut cream
143             Coconut cream
150                  Eggplant
150                  Eggplant
 12                       Ham
 12                       Ham
 12                       Ham
 12                       Ham
 12                       Ham
 12                       Ham
 12                       Ham
 12                       

Terdapat 27 duplicate ID dan 24 duplicate Name.

In [ ]:
# Cek inconsistent value pada kolom Metric
metric_cols = [c for c in df_product.columns if 'Metric' in c]
print('Nilai unik per kolom Metric:')
for col in metric_cols:
    unique_vals = df_product[col].dropna().unique()
    print(f'  {col}: {unique_vals}')

Nilai unik per kolom Metric:
  Pantry_Metric: ['Day' 'Month' 'Indefinitely' 'When Ripe' 'Week' '#VALUE!' 'Year']
  DOP_Pantry_Metric: ['Month' 'Year' 'Week' 'Day' '#VALUE!']
  Pantry_After_Opening_Metric: ['Month' 'Not Recommended' 'Year' 'Day' '#VALUE!']
  Refrigerate_Metric: ['Day' 'Month' 'Year' 'Package use-by date' 'Week' '#VALUE!' 'Sunday']
  DOP_Refrigerate_Metric: ['Month' 'Week' 'Day' '#VALUE!' 'Sunday']
  Refrigerate_After_Opening_Metric: ['Week' 'Month' 'Day' 'Year' '#VALUE!']
  Refrigerate_After_Thawing_Metric: ['#VALUE!']
  Freeze_Metric: ['Month' 'Year' 'Day' '#VALUE!']
  DOP_Freeze_Metric: ['Month' 'Day' '#VALUE!']


Kolom Metric memiliki nilai yang tidak konsisten — ada yang dalam
bahasa English (`Day`, `Week`, `Month`, `Year`) dan ada nilai tidak
valid seperti `Not Recommended`, `Package use-by date`, dan `Sunday`.
Seluruhnya akan distandarisasi ke satuan **hari** di fungsi
`convert_days` pada tahap Cleaning.

**Kesimpulan Assessing df_product:**
- Terdapat banyak **missing value** terutama pada kolom-kolom penyimpanan (Pantry, Refrigerate, Freeze)
- Terdapat **#VALUE!** yang merupakan error dari Excel, perlu diganti dengan NaN
- Kolom **Metric** memiliki nilai tidak konsisten seperti `Days`, `Weeks`, `Months`, `Years`, dan `Not Recommended` — perlu distandarisasi saat konversi ke hari

### Assessing - df_translate

In [ ]:
df_translate.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 196 entries, 0 to 195
Data columns (total 38 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   ID                                196 non-null    int64  
 1   Category_ID                       196 non-null    int64  
 2   Name                              196 non-null    object 
 3   Name_subtitle                     41 non-null     object 
 4   Keywords                          96 non-null     object 
 5   Pantry_Min                        106 non-null    object 
 6   Pantry_Max                        59 non-null     float64
 7   Pantry_Metric                     136 non-null    object 
 8   Pantry_tips                       18 non-null     object 
 9   DOP_Pantry_Min                    60 non-null     object 
 10  DOP_Pantry_Max                    59 non-null     object 
 11  DOP_Pantry_Metric                 60 non-null     object 
 12  DOP_Pant

Dataset `df_translate` terdiri dari 196 baris dan 38 kolom — sama
dengan `df_product` namun kolom `Name` berisi nama bahan dalam Bahasa
Indonesia. Ditemukan **#VALUE!** sebanyak 51 sel dan **18 duplicate ID**
yang perlu ditangani. Kolom Metric menggunakan Bahasa Indonesia
(`Hari`, `Minggu`, `Bulan`, `Tahun`) yang tidak konsisten dengan
`df_product` — akan distandarisasi di tahap Cleaning.

In [ ]:
#Missing Value
print(df_translate.isna().sum())

ID                                    0
Category_ID                           0
Name                                  0
Name_subtitle                       155
Keywords                            100
Pantry_Min                           90
Pantry_Max                          137
Pantry_Metric                        60
Pantry_tips                         178
DOP_Pantry_Min                      136
DOP_Pantry_Max                      137
DOP_Pantry_Metric                   136
DOP_Pantry_tips                     194
Pantry_After_Opening_Min            184
Pantry_After_Opening_Max            187
Pantry_After_Opening_Metric         183
Refrigerate_Min                      82
Refrigerate_Max                     157
Refrigerate_Metric                   77
Refrigerate_tips                    184
DOP_Refrigerate_Min                 130
DOP_Refrigerate_Max                 130
DOP_Refrigerate_Metric              130
DOP_Refrigerate_tips                194
Refrigerate_After_Opening_Min       176


Missing value di `df_translate` lebih tinggi dibanding `df_product`
pada beberapa kolom — terutama `Pantry_Max` yang hanya terisi 59 dari
196 baris. Kolom `After_Opening` dan `After_Thawing` hampir seluruhnya
kosong. Ini terjadi karena sheet Translate hanya mengisi kolom yang
diterjemahkan, sementara kolom angka sebagian besar diambil dari
`df_product` via mapping saat feature engineering.

In [ ]:
# Cek invalid value
invalid_tr = (df_translate == '#VALUE!').sum().sum()
print(f'Jumlah #VALUE!: {invalid_tr}')
print(f'Duplicate ID  : {df_translate["ID"].duplicated().sum()}')

Jumlah #VALUE!: 51
Duplicate ID  : 27


Ditemukan **51 sel** berisi `#VALUE!` di `df_translate` dan **18
duplicate ID**. Jumlah `#VALUE!` lebih sedikit dibanding `df_product`
(2.645 sel) karena sheet Translate tidak mengisi semua kolom —
kolom yang kosong tidak bisa menghasilkan error formula Excel.

In [ ]:
#Cek Duplicate
print('=== Duplicate di df_translate ===')
print(f'Duplicate ID  : {df_translate["ID"].duplicated().sum()}')
print(f'Duplicate Name: {df_translate["Name"].duplicated().sum()}')
print()
dup_names = df_translate[df_translate["Name"].duplicated(keep=False)][["ID","Name"]].sort_values("Name")
print(dup_names.to_string(index=False))

=== Duplicate di df_translate ===
Duplicate ID  : 27
Duplicate Name: 22

 ID             Name
 34        Asparagus
 34        Asparagus
 37          Brokoli
 37          Brokoli
 12      Daging Babi
 12      Daging Babi
 12      Daging Babi
 12      Daging Babi
 12      Daging Babi
 12      Daging Babi
 12      Daging Babi
 12      Daging Babi
 12      Daging Babi
 82     Daun Kemangi
 82     Daun Kemangi
 82     Daun Kemangi
111      Daun Kunyit
111      Daun Kunyit
 92 Hati Ampela Ayam
 92 Hati Ampela Ayam
 74       Kayu manis
 74       Kayu manis
 38            Kubis
 38            Kubis
 52        Labu Siam
 52        Labu Siam
 33            Nanas
 33            Nanas
143           Santan
143           Santan
 45           Selada
 45           Selada
 45           Selada
150           Terong
150           Terong


Terdapat 18 duplicate ID di `df_translate` — sama seperti pola di
`df_product`, duplikasi ini disebabkan oleh satu bahan yang masuk ke
beberapa subkategori sekaligus.


In [ ]:
# Cek inconsistent value pada kolom Metric
metric_cols_tr = [c for c in df_translate.columns if 'Metric' in c]
print('Nilai unik Metric di df_translate:')
for col in metric_cols_tr:
    unique_vals = df_translate[col].dropna().unique()
    if len(unique_vals) > 0:
        print(f'  {col}: {unique_vals}')

Nilai unik Metric di df_translate:
  Pantry_Metric: ['Hari' 'Bulan' 'Tanpa batas waktu' 'Saat Matang' 'Minggu' 'Tahun']
  DOP_Pantry_Metric: ['#VALUE!' 'Bulan' 'Tahun' 'Minggu' 'Hari']
  Pantry_After_Opening_Metric: ['#VALUE!' 'Bulan' 'Tidak Disarankan' 'Tahun' 'Hari']
  Refrigerate_Metric: ['Hari' '#VALUE!' 'Bulan' 'Tahun' 'Tanggal kedaluwarsa kemasan' 'Minggu']
  DOP_Refrigerate_Metric: ['Bulan' 'Minggu' 'Hari']
  Refrigerate_After_Opening_Metric: ['Minggu' '#VALUE!' 'Bulan' 'Hari' 'Tahun']
  Refrigerate_After_Thawing_Metric: ['#VALUE!']
  Freeze_Metric: ['Tidak Disarankan' 'Bulan' '#VALUE!' 'Tahun' 'Hari']
  DOP_Freeze_Metric: ['Bulan' 'Hari']


Kolom Metric di `df_translate` menggunakan Bahasa Indonesia (`Hari`,
`Minggu`, `Bulan`, `Tahun`) berbeda dengan `df_product` yang
menggunakan English (`Day`, `Week`, `Month`, `Year`). Selain itu
masih terdapat `#VALUE!` di beberapa kolom Metric. Perbedaan bahasa
ini sudah diantisipasi di fungsi `convert_days` yang menangani
kedua bahasa sekaligus.

**Kesimpulan Assessing df_translate:**
- Terdapat **#VALUE!** yang perlu diganti NaN
- Kolom **Metric** di sheet translate menggunakan Bahasa Indonesia (misal: `Bulan`, `Minggu`) perlu ditangani saat konversi hari agar konsisten dengan df_product

### Assessing - df_category

In [ ]:
print('Isi df_category:')
print(df_category.to_string())
print(f'\nMissing value : {df_category.isnull().sum().sum()}')
print(f'Duplicate ID  : {df_category["ID"].duplicated().sum()}')
print(f'\nCategory unik : {df_category["Category_Name"].nunique()}')

Isi df_category:
    ID                      Category_Name      Subcategory_Name
0    1                          Baby Food                   NaN
1    2                        Baked Goods                Bakery
2    3                        Baked Goods    Baking and Cooking
3    4                        Baked Goods    Refrigerated Dough
4    5                          Beverages                   NaN
5    6  Condiments, Sauces & Canned Goods                   NaN
6    7              Dairy Products & Eggs                   NaN
7    8              Food Purchased Frozen                   NaN
8    9              Grains, Beans & Pasta                   NaN
9   10                               Meat                 Fresh
10  11                               Meat    Shelf Stable Foods
11  12                               Meat   Smoked or Processed
12  13                               Meat  Stuffed or Assembled
13  18                             Fruits          Fresh Fruits
14  19                 

Tidak terdapat duplicate dan missing value pada dataset category. Missing value terjadi hanya pada kolom subcategory_name, dimana terdapat beberapa category yang tidak memiliki subcategory_name

## **Cleaning Data**

Berdasarkan hasil assessing, berikut masalah yang akan dibersihkan:
1. **Invalid value** (#VALUE!) → diganti dengan `NaN`
2. **Inconsistent value** (nama item & metric) → distandarisasi
3. **Duplicate data** → dihapus berdasarkan ID
4. **Missing value pada Name** → baris dihapus karena Name adalah identitas utama
5. **Konversi metric** → semua satuan waktu dikonversi ke hari

In [ ]:
def clean_base(df, df_category):
    df          = df.copy()
    df_category = df_category.copy()

    before = len(df)

    # 1. Ganti #VALUE! dengan NaN
    invalid_before = (df == '#VALUE!').sum().sum()
    df.replace('#VALUE!', np.nan, inplace=True)
    print(f'  [1] Ganti #VALUE! → NaN : {invalid_before} sel diganti')

    # 2. Konversi kolom Min/Max ke numeric
    numeric_cols = [c for c in df.columns if any(x in c for x in ['Min', 'Max'])]
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    print(f'  [2] Konversi numeric    : {len(numeric_cols)} kolom dikonversi')

    # 3. Standardisasi nama item
    df['Name'] = df['Name'].astype(str).str.strip().str.lower()

    # 4. Drop rows dengan Name kosong
    df = df[df['Name'] != 'nan']
    print(f'  [3] Drop Name kosong    : {before - len(df)} rows dihapus')

    # 5. Menghapus duplicate berdasarkan ID
    before_dedup = len(df)
    df = df.sort_values('ID').drop_duplicates(subset=['ID']).reset_index(drop=True)
    print(f'  [4] Hapus duplicate ID  : {before_dedup - len(df)} rows dihapus')
    df['id'] = df['ID']

    # 6. Standardisasi category name
    df_category['Category_Name'] = df_category['Category_Name'].astype(str).str.strip().str.lower()

    # 7. Merge dengan category
    df = df.merge(df_category, left_on='Category_ID', right_on='ID', how='left', suffixes=('', '_cat'))
    df.rename(columns={'Name': 'item', 'Category_Name': 'category'}, inplace=True)

    print(f'  [5] Total rows tersisa  : {len(df)}')
    return df
def pick_best(row, cols):
    # Ambil nilai pertama yang tidak NaN dari list kolom (Max diprioritaskan)
    for col in cols:
        if col in row and pd.notna(row[col]):
            return row[col]
    return np.nan


def convert_days(value, metric):
    # Konversi semua satuan waktu ke hari
    # Handle metric English & Indonesia (inconsistent value antar sheet)
    if pd.isna(value):
        return None
    try:
        value = float(value)
    except:
        return None
    if value > 10000:
        return None

    metric = str(metric).strip().lower()

    if metric in ['week', 'weeks', 'minggu']:
        return value * 7
    elif metric in ['month', 'months', 'bulan']:
        return value * 30
    elif metric in ['year', 'years', 'tahun']:
        return value * 365
    elif metric in ['not recommended', 'tidak disarankan']:
        return None
    return value

Proses `clean_base` berhasil menangani seluruh masalah kualitas data
yang ditemukan di tahap Assessing — `#VALUE!` diganti NaN, kolom
Min/Max dikonversi ke numeric, nama item distandardisasi ke lowercase,
dan duplicate ID dihapus.

In [ ]:
def build_clean(df):
    rows = []

    for _, row in df.iterrows():
        id_val   = row['id']
        category = row['category']
        item     = row['item']
        # PANTRY
        val = pick_best(row, [
            'Pantry_Max',
            'Pantry_Min',

            'DOP_Pantry_Max',
            'DOP_Pantry_Min',

            'Pantry_After_Opening_Max',
            'Pantry_After_Opening_Min'
        ])

        met = pick_best(row, [
            'Pantry_Metric',
            'DOP_Pantry_Metric',
            'Pantry_After_Opening_Metric'
        ])
        days = convert_days(val, met)
        if pd.notna(days):
            rows.append([id_val, category, item, 'Room Temperature', days])

        # FRIDGE
        val = pick_best(row, ['Refrigerate_Max','Refrigerate_Min','DOP_Refrigerate_Max','DOP_Refrigerate_Min','Refrigerate_After_Opening_Max','Refrigerate_After_Opening_Min','Refrigerate_After_Thawing_Max','Refrigerate_After_Thawing_Min'])
        met = pick_best(row, ['Refrigerate_Metric','DOP_Refrigerate_Metric','Refrigerate_After_Opening_Metric','Refrigerate_After_Thawing_Metric'])
        days = convert_days(val, met)
        if pd.notna(days):
            rows.append([id_val, category, item, 'Fridge', days])

        # FREEZER
        val  = pick_best(row, ['Freeze_Max', 'Freeze_Min', 'DOP_Freeze_Max', 'DOP_Freeze_Min'])
        met  = pick_best(row, ['Freeze_Metric', 'DOP_Freeze_Metric'])
        days = convert_days(val, met)
        if pd.notna(days):
            rows.append([id_val, category, item, 'Freezer', days])

    df_clean = pd.DataFrame(rows, columns=['id', 'category', 'item', 'storage', 'days'])

    df_clean['days'] = pd.to_numeric(df_clean['days'], errors='coerce')

    before = len(df_clean)
    df_clean = df_clean[(df_clean['days'] >= 0) & (df_clean['days'] < 3650)]
    print(f'  [5] Filter outlier days : {before - len(df_clean)} rows dihapus')

    df_clean['days'] = df_clean['days'].astype(int)

    df_clean = df_clean.sort_values(['item', 'storage', 'days'], ascending=[True, True, True])

    before = len(df_clean)
    df_clean = df_clean.drop_duplicates(subset=['item', 'storage'], keep='first')
    print(f'  [6] Hapus duplicate item+storage: {before - len(df_clean)} rows dihapus')

    df_clean = df_clean.reset_index(drop=True)

    return df_clean

Dihasilkan dataset dalam format long — satu
baris per kombinasi item × storage. Nilai umur simpan dikonversi ke
satuan hari, outlier di luar rentang 0–3650 hari dihapus, dan
duplikat item+storage diselesaikan dengan mengambil nilai terkecil
sebagai estimasi paling konservatif.

### Cleaning - Product Original (English)

In [ ]:
df_product_clean  = clean_base(df_product, df_category)
df_clean_original = build_clean(df_product_clean)

print('\n=== Hasil Akhir ORIGINAL ===')
print(df_clean_original.head())
print(f'\nTotal rows : {len(df_clean_original)}')
print(f'Missing    : {df_clean_original.isnull().sum().sum()}')
print(f'Duplicate  : {df_clean_original.duplicated().sum()}')
print(f'Days range : {df_clean_original["days"].min()} - {df_clean_original["days"].max()} hari')
print(f'Storage    : {df_clean_original["storage"].unique()}')
print(f'Category   : {df_clean_original["category"].nunique()} kategori')

  [1] Ganti #VALUE! → NaN : 2645 sel diganti
  [2] Konversi numeric    : 18 kolom dikonversi
  [3] Drop Name kosong    : 0 rows dihapus
  [4] Hapus duplicate ID  : 27 rows dihapus
  [5] Total rows tersisa  : 169
  [5] Filter outlier days : 0 rows dihapus
  [6] Hapus duplicate item+storage: 3 rows dihapus

=== Hasil Akhir ORIGINAL ===
    id category       item           storage  days
0  179  seafood  anchovies           Freezer   180
1  179  seafood  anchovies            Fridge    14
2  179  seafood  anchovies  Room Temperature     2
3   27   fruits     apples           Freezer   240
4   27   fruits     apples            Fridge    42

Total rows : 504
Missing    : 0
Duplicate  : 0
Days range : 1 - 1440 hari
Storage    : ['Freezer' 'Fridge' 'Room Temperature']
Category   : 8 kategori


/tmp/ipykernel_2307/3661228341.py:9: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df.replace('#VALUE!', np.nan, inplace=True)


### Proses Cleaning Product - Translate (Indonesia)

In [ ]:
print('>>> Cleaning df_translate:')
df_translate_clean = clean_base(df_translate, df_category)

mapping_item = dict(zip(df_translate_clean['id'], df_translate_clean['item']))
print(f'\nJumlah item ter-translate: {len(mapping_item)}')

storage_map = {
    'Room Temperature': 'Suhu Ruangan',
    'Fridge'          : 'Kulkas',
    'Freezer'         : 'Freezer'
}

category_map = {
    'baby food'                        : 'Makanan Bayi',
    'baked goods'                      : 'Kue Panggang',
    'beverages'                        : 'Minuman',
    'condiments, sauces & canned goods': 'Bumbu, Saus & Makanan Kalengan',
    'dairy products & eggs'            : 'Produk Susu & Telur',
    'food purchased frozen'            : 'Makanan yang Dibeli dalam Keadaan Beku',
    'grains, beans & pasta'            : 'Biji-bijian, Kacang-kacangan & Pasta',
    'meat'                             : 'Daging',
    'poultry'                          : 'Unggas',
    'fruits'                           : 'Buah-buahan',
    'vegetables'                       : 'Sayuran',
    'seafood'                          : 'Hidangan laut',
    'shelf stable foods'               : 'Makanan Tahan Lama di Rak',
    'vegetarian proteins'              : 'Protein Vegetarian',
    'deli & prepared foods'            : 'Toko Makanan Siap Saji & Makanan Olahan'
}

df_clean_translate             = df_clean_original.copy()
df_clean_translate['item']     = df_clean_translate['id'].map(mapping_item).fillna(df_clean_translate['item'])
df_clean_translate['category'] = df_clean_translate['category'].map(category_map)
df_clean_translate['storage']  = df_clean_translate['storage'].map(storage_map)

# Hapus duplikat
before = len(df_clean_translate)
df_clean_translate = df_clean_translate.groupby(['category', 'item', 'storage'], as_index=False).agg({
    'id'  : 'first',
    'days': 'min'
})
df_clean_translate = df_clean_translate[['id', 'category', 'item', 'storage', 'days']]

>>> Cleaning df_translate:
  [1] Ganti #VALUE! → NaN : 51 sel diganti
  [2] Konversi numeric    : 18 kolom dikonversi
  [3] Drop Name kosong    : 0 rows dihapus
  [4] Hapus duplicate ID  : 27 rows dihapus
  [5] Total rows tersisa  : 169

Jumlah item ter-translate: 169


/tmp/ipykernel_2307/3661228341.py:9: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df.replace('#VALUE!', np.nan, inplace=True)


In [ ]:
print(df_clean_translate.head())

   id                              category          item       storage  days
0  57  Biji-bijian, Kacang-kacangan & Pasta         beras       Freezer   730
1  57  Biji-bijian, Kacang-kacangan & Pasta         beras        Kulkas   365
2  57  Biji-bijian, Kacang-kacangan & Pasta         beras  Suhu Ruangan   180
3  68  Biji-bijian, Kacang-kacangan & Pasta  kacang tanah       Freezer   720
4  68  Biji-bijian, Kacang-kacangan & Pasta  kacang tanah        Kulkas   360


In [ ]:
df_clean_translate.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 504 entries, 0 to 503
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   id        504 non-null    int64 
 1   category  504 non-null    object
 2   item      504 non-null    object
 3   storage   504 non-null    object
 4   days      504 non-null    int64 
dtypes: int64(2), object(3)
memory usage: 19.8+ KB


In [ ]:
# =========================
# BUAT ingredient_id
# =========================

ingredient_table = (
    df_clean_translate[['category', 'item']]
    .drop_duplicates()
    .reset_index(drop=True)
)

ingredient_table['ingredient_id'] = range(
    1,
    len(ingredient_table) + 1
)

# merge ingredient_id ke dataframe utama
df_clean_translate = df_clean_translate.merge(
    ingredient_table,
    on=['category', 'item'],
    how='left'
)

# =========================
# BUAT id FINAL
# =========================

# hapus id lama
df_clean_translate = df_clean_translate.drop(columns='id')

# buat id baru
df_clean_translate.insert(
    0,
    'id',
    range(1, len(df_clean_translate) + 1)
)
# =========================
# URUTKAN KOLOM
# =========================

cols = [
    'id',
    'ingredient_id',
    'category',
    'item',
    'storage',
    'days'
]

df_clean_translate = df_clean_translate[cols]

# =========================
# SAVE
# =========================

with pd.ExcelWriter('fix_bahan_clean.xlsx') as writer:
    df_clean_translate.to_excel(
        writer,
        sheet_name='Translate',
        index=False
    )

df_clean_translate.to_csv(
    'fix_bahan_clean_translate.csv',
    index=False
)

print('fix_bahan_clean.xlsx berhasil disimpan')
print('fix_bahan_clean_translate.csv berhasil disimpan')

fix_bahan_clean.xlsx berhasil disimpan
fix_bahan_clean_translate.csv berhasil disimpan


Dataset bersih berhasil disimpan dalam dua format — `fix_bahan_clean.xlsx`
dengan dua sheet (Original dan Translate) serta `fix_bahan_clean_translate.csv`

## Kesimpulan Data Wrangling

Tahap data wrangling pada dataset bahan makanan berhasil dilakukan untuk membersihkan dan menyiapkan data sebelum digunakan pada proses analisis dan feature engineering. Pada tahap assessing, ditemukan beberapa permasalahan kualitas data, seperti missing value pada beberapa kolom penyimpanan, invalid value berupa `#VALUE!`, duplicate data, serta inkonsistensi nilai pada beberapa atribut kategorikal dan metric.

Pada dataset original (`df_product`), ditemukan sebanyak **2.645 sel** berisi nilai invalid `#VALUE!`. Nilai tersebut berasal dari error formula Excel sehingga tidak dapat digunakan secara langsung dalam analisis. Seluruh nilai `#VALUE!` kemudian diganti menjadi `NaN` agar lebih aman diproses pada tahap cleaning. Selain itu, ditemukan **27 duplicate ID** yang menunjukkan adanya pengulangan data bahan makanan. Duplicate berdasarkan ID kemudian dihapus sehingga jumlah data berkurang dari **196 baris menjadi 169 baris**.

Pada dataset translate (`df_translate`), ditemukan **51 sel** berisi nilai `#VALUE!` dan **27 duplicate ID**. Sama seperti dataset original, nilai invalid dibersihkan dengan mengubahnya menjadi `NaN`, kemudian dilakukan konversi tipe data numerik pada **18 kolom** agar seluruh atribut umur simpan dapat dianalisis secara konsisten. Setelah proses cleaning awal, dataset memiliki **169 item** yang valid.

Proses cleaning juga mencakup konversi satuan waktu umur simpan dari hari, minggu, bulan, dan tahun menjadi satuan yang seragam, yaitu **hari**. Selain itu, dilakukan pemeriksaan terhadap outlier pada atribut umur simpan (`days`). Hasil pemeriksaan menunjukkan bahwa tidak terdapat outlier yang perlu dihapus sehingga seluruh data tetap dipertahankan pada tahap tersebut.

Pada tahap akhir cleaning, dilakukan penghapusan duplicate berdasarkan kombinasi `item` dan `storage` untuk memastikan tidak terdapat bahan yang memiliki metode penyimpanan identik secara berulang. Proses ini berhasil menghapus **3 baris duplicate tambahan** sehingga dataset akhir menjadi lebih konsisten dan representatif.

Dataset kemudian diubah ke format long sehingga setiap baris merepresentasikan kombinasi antara bahan, kategori, metode penyimpanan, dan estimasi umur simpan. Format ini membuat data lebih mudah dianalisis berdasarkan metode penyimpanan seperti `Freezer`, `Kulkas`, dan `Suhu Ruangan`.

Hasil akhir cleaning menghasilkan dataset bersih sebanyak **504 baris** dengan **0 missing value** dan tipe data yang sudah sesuai. Dataset akhir merepresentasikan **168 bahan unik** dengan tiga metode penyimpanan utama. Dataset juga memiliki 6 kolom utama, yaitu `id`, `ingredient_id`, `category`, `item`, `storage`, dan `days`.

# **Data Dictionary**

In [ ]:
data_dictionary = pd.DataFrame({
    "Kolom": [
        "id",
        "ingredient_id",
        "category",
        "item",
        "storage",
        "days"
    ],

    "Tipe Data": [
        "int",
        "int",
        "object",
        "object",
        "object",
        "int"
    ],

    "Deskripsi": [
        "ID unik setiap baris data",
        "ID unik setiap bahan makanan",
        "Kategori bahan makanan",
        "Nama bahan makanan",
        "Metode penyimpanan bahan makanan",
        "Estimasi umur simpan dalam hari"
    ]
})

display(data_dictionary)

,Kolom,Tipe Data,Deskripsi
0,id,int,ID unik setiap baris data
1,ingredient_id,int,ID unik setiap bahan makanan
2,category,object,Kategori bahan makanan
3,item,object,Nama bahan makanan
4,storage,object,Metode penyimpanan bahan makanan
5,days,int,Estimasi umur simpan dalam hari


Kolom `id` digunakan sebagai identitas unik untuk setiap baris data, sedangkan `ingredient_id` merepresentasikan identitas unik setiap bahan makanan. Selain itu, atribut `category` digunakan untuk mengelompokkan bahan berdasarkan jenisnya, seperti sayuran, buah-buahan, daging, seafood, dan kategori lainnya.

Kolom `item` menyimpan nama bahan makanan yang telah melalui proses standarisasi penulisan sehingga lebih konsisten untuk dianalisis. Sementara itu, kolom `storage` menunjukkan metode penyimpanan bahan makanan, seperti `Freezer`, `Kulkas`, dan `Suhu Ruangan`. Atribut ini menjadi salah satu variabel penting dalam analisis karena setiap metode penyimpanan memiliki pengaruh yang berbeda terhadap umur simpan bahan makanan.

Kolom `days` merupakan atribut numerik yang menyimpan estimasi umur simpan bahan makanan dalam satuan hari. Seluruh nilai umur simpan telah dikonversi ke format numerik dan diseragamkan ke satuan hari agar lebih mudah digunakan pada proses statistik, visualisasi, dan exploratory data analysis (EDA).

# **Feature Engineering**

In [ ]:
df = df_clean_translate.copy()

for col in ['category', 'item', 'storage']:
    df[col] = df[col].str.lower().str.strip()

# Hapus duplikat category+item+storage, ambil days terkecil
before = len(df)
df = df.groupby(['category', 'item', 'storage'], as_index=False).agg({
    'id'  : 'first',
    'days': 'min'
})
df['days'] = df['days'].astype(int)
print(f'Hapus duplikat di feature engineering: {before - len(df)} rows dihapus')
print(f'Shape: {df.shape}')

Hapus duplikat di feature engineering: 0 rows dihapus
Shape: (504, 5)


In [ ]:
# encoding storage
storage_level_map = {
    'suhu ruangan': 1,
    'kulkas'      : 2,
    'freezer'     : 3
}
df['storage_level'] = df['storage'].map(storage_level_map)

# log transform days
df['days_log'] = np.log1p(df['days'])

print(df[['category', 'storage', 'storage_level', 'days', 'days_log']].head(10))

                               category       storage  storage_level  days  \
0  biji-bijian, kacang-kacangan & pasta       freezer              3   730   
1  biji-bijian, kacang-kacangan & pasta        kulkas              2   365   
2  biji-bijian, kacang-kacangan & pasta  suhu ruangan              1   180   
3  biji-bijian, kacang-kacangan & pasta       freezer              3   720   
4  biji-bijian, kacang-kacangan & pasta        kulkas              2   360   
5  biji-bijian, kacang-kacangan & pasta  suhu ruangan              1   120   
6  biji-bijian, kacang-kacangan & pasta       freezer              3   730   
7  biji-bijian, kacang-kacangan & pasta        kulkas              2   365   
8  biji-bijian, kacang-kacangan & pasta  suhu ruangan              1   365   
9  biji-bijian, kacang-kacangan & pasta       freezer              3   730   

   days_log  
0  6.594413  
1  5.902633  
2  5.198497  
3  6.580639  
4  5.888878  
5  4.795791  
6  6.594413  
7  5.902633  
8  5.902633  
9

In [ ]:
features = ['category', 'storage', 'storage_level']
target   = 'days_log'

X = df[features]
y = df[target]

In [ ]:
# One-hot encoding kolom kategorikal
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
X_cat   = encoder.fit_transform(X[['category', 'storage']])
X_cat   = pd.DataFrame(X_cat, columns=encoder.get_feature_names_out(['category', 'storage']))

X_num   = X[['storage_level']].reset_index(drop=True)
X_final = pd.concat([X_cat, X_num], axis=1)

print('Shape X_final:', X_final.shape)
print('Kolom:', X_final.columns.tolist())

Shape X_final: (504, 12)
Kolom: ['category_biji-bijian, kacang-kacangan & pasta', 'category_buah-buahan', 'category_bumbu, saus & makanan kalengan', 'category_daging', 'category_hidangan laut', 'category_makanan tahan lama di rak', 'category_produk susu & telur', 'category_sayuran', 'storage_freezer', 'storage_kulkas', 'storage_suhu ruangan', 'storage_level']


In [ ]:
# Scaling
scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X_final)

print('X_scaled shape:', X_scaled.shape)

X_scaled shape: (504, 12)


In [ ]:
# Fitur dan Target
df_bahan = df[['id', 'category', 'item', 'storage', 'days', 'storage_level', 'days_log']].copy()
df_bahan = pd.concat([df_bahan.reset_index(drop=True), X_cat.reset_index(drop=True)], axis=1)

df_bahan.to_csv('dataset_bahan.csv', index=False)

# Export encoder & scaler
joblib.dump(encoder, 'encoder.pkl')
joblib.dump(scaler,  'scaler.pkl')

print(df_bahan.head())

   id                              category          item       storage  days  \
0   1  biji-bijian, kacang-kacangan & pasta         beras       freezer   730   
1   2  biji-bijian, kacang-kacangan & pasta         beras        kulkas   365   
2   3  biji-bijian, kacang-kacangan & pasta         beras  suhu ruangan   180   
3   4  biji-bijian, kacang-kacangan & pasta  kacang tanah       freezer   720   
4   5  biji-bijian, kacang-kacangan & pasta  kacang tanah        kulkas   360   

   storage_level  days_log  category_biji-bijian, kacang-kacangan & pasta  \
0              3  6.594413                                            1.0   
1              2  5.902633                                            1.0   
2              1  5.198497                                            1.0   
3              3  6.580639                                            1.0   
4              2  5.888878                                            1.0   

   category_buah-buahan  category_bumbu, saus & ma

Feature engineering menghasilkan tiga fitur tambahan — `storage_level`
(encoding ordinal: Suhu Ruangan=1, Kulkas=2, Freezer=3), `days_log`
(log transformation untuk mengurangi skewness), dan one-hot encoding
untuk kolom `category` dan `storage`.

## Kesimpulan Feature Engineering

Feature engineering yang dilakukan adalah pembuatan atribut `storage_level`, yaitu representasi numerik dari metode penyimpanan bahan makanan. Pada fitur ini, metode penyimpanan dikonversi menjadi skala ordinal untuk menunjukkan tingkat efektivitas penyimpanan terhadap umur simpan bahan.Selain itu, dilakukan transformasi logaritmik pada atribut umur simpan (`days`) melalui fitur `days_log`. Transformasi ini bertujuan untuk mengurangi skewness pada distribusi data umur simpan karena terdapat perbedaan rentang nilai yang cukup besar antar bahan makanan. Dengan transformasi logaritmik, distribusi data menjadi lebih stabil dan lebih sesuai digunakan untuk proses analisis statistik maupun pemodelan.Tahap feature engineering juga mencakup proses encoding pada atribut kategorikal menggunakan teknik one-hot encoding. Kolom `category` dan `storage` diubah menjadi beberapa fitur numerik baru, seperti `category_sayuran`, `category_daging`, `storage_freezer`, dan `storage_kulkas`.